# 260408제너레이터

## 예제 1

In [1]:
import tracemalloc
import time

def report(label):
    #현재와 최대 메모리 사용량을 바이트 단위로 변환
    current, peak = tracemalloc.get_traced_memory()
    print(f"{label} | 현재={current/1024/1024:8.2f} MiB | 최대 {peak/1024/1024:8.2f} Mib")

N = 5_000_000
tracemalloc.start()
report("시작")

# 1) meterialize : 리스트로 한번에 만들기
t0 = time.perf_counter()
lst = [i for i in range(N)]
report("리스트 컴프리센션 생성 후")
print(f"리스트 생성 시간: {time.perf_counter() - t0:.3f}초")

# 리스트를 지우고 GC 타이밍 영향을 줄이기 위해 참조 제거
lst = None
report("리스트 제거 후")

# 2) lazy: 제너레이터는 만들기만 하면 거의 할당이 없다.
t0 = time.perf_counter()
gen = (i for i in range(N))
report("제너에이터 생성 후")
print(f"제너레이터 생성 시간: {time.perf_counter() - t0:.3f}초")

# 다만 "소비"하면 그때 계산이 진행됨 (여기서부터는 누적합으로 소비)
t0 = time.perf_counter()
s = sum(gen)
report("제너레이터 소비 후")
print(f"제너레이터 소비 시간: {time.perf_counter() - t0:.3f}초")
print("합계:", s)

tracemalloc.stop()

시작 | 현재=    0.00 MiB | 최대     0.02 Mib
리스트 컴프리센션 생성 후 | 현재=  194.49 MiB | 최대   194.51 Mib
리스트 생성 시간: 1.435초
리스트 제거 후 | 현재=    0.01 MiB | 최대   194.52 Mib
제너에이터 생성 후 | 현재=    0.01 MiB | 최대   194.52 Mib
제너레이터 생성 시간: 0.000초
제너레이터 소비 후 | 현재=    0.00 MiB | 최대   194.52 Mib
제너레이터 소비 시간: 1.150초
합계: 12499997500000


## 예제 2

In [2]:
my_str = 'abc'
my_iter = iter(my_str)

print(next(my_iter))
print(next(my_iter))
print(next(my_iter))
print(next(my_iter))

a
b
c


StopIteration: 

In [ ]:
class CountUp:
    def __init__(self, start, end):
        self.cur = start
        self.end = end

    # __iter__는 iterable 객체에서 iterator 객체를 반환하는 메서드
    def __iter__(self):
        return self # iterator == iterable
    
    def __next__(self):
        # StopIteration 예외를 발생시키면 for 루프가 종료됨
        if self.cur > self.end:
            raise StopIteration
        v = self.cur
        self.cur += 1
        return v
    
for x in CountUp(3, 7):
    print(x)

3
4
5
6
7


## 예제 3

In [3]:
def count_up(start, end):
    cur = start
    while cur <= end:
        yield cur
        cur += 1

for x in count_up(3, 7):
    print(x)

3
4
5
6
7


## 예제 4

In [4]:
def infinite_sequence():
    num = 0
    while True:
        yield num
        num += 1

gen = infinite_sequence()
print(next(gen)) # 0
print(next(gen)) # 1
print(next(gen)) # 2

0
1
2


## 예제 5

In [6]:
# 피보나치 수열의 무한 시퀀스
def fibonacci_generator():
    n1, n2 = 0, 1
    while True:
        yield n1
        n1, n2 = n2, n1 + n2

gen = fibonacci_generator()

#첫 10개의 피보나치 수를 출력
for _ in range(10):
    print(next(gen)) # gen.__next__()와 동일

print(next(gen)) # 11번째 피보나치 수
print(next(gen)) # 12번째 피보나치 수

0
1
1
2
3
5
8
13
21
34
55
89


## 예제 6

In [7]:
def read_large_file_without_generator(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()
    return [line.strip() for line in lines]

file_path = 'large_data_file.txt'

# 전체 파일명을 메모리에 로드하여 처리
lines = read_large_file_without_generator(file_path)

for line in lines:
    print(line)

FileNotFoundError: [Errno 2] No such file or directory: 'large_data_file.txt'

In [ ]:
def read_large_file_with_generator(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            yield line.strip()

#예제 파일 경로
file_path = 'large_data_file.txt'

# 전체 파일명을 메모리에 로드하여 처리
lines = read_large_file_with_generator(file_path)

for line in lines:
    print(line)

## 예제 7

In [8]:
def wrapper_with_yield_from(xs):
    yield from xs

data = [1, 2, 3]
print(list(wrapper_with_yield_from(data)))

[1, 2, 3]


In [10]:
def a():
    yield [1,2,3]

def b():
    yield from [1,2,3]

print(list(a()))
print(list(b()))

[[1, 2, 3]]
[1, 2, 3]


In [12]:
import inspect

def gen():
    print('A, 시작')
    x = 1
    yield x
    print('B, 중간')
    x += 1
    yield x
    print('C, 끝')

g = gen()
print('상태:', inspect.getgeneratorstate(g)) # GEN_CREATED

print('next ->', next(g))
print('상태:', inspect.getgeneratorstate(g)) # GEN_SUSPENDED

print('next ->', next(g))
print('상태:', inspect.getgeneratorstate(g)) # GEN_SUSPENDED

try:
    print('next ->', next(g))
except StopIteration:
    print('StopIteration raised')
print('상태:', inspect.getgeneratorstate(g)) # GEN_CLOSED

상태: GEN_CREATED
A, 시작
next -> 1
상태: GEN_SUSPENDED
B, 중간
next -> 2
상태: GEN_SUSPENDED
C, 끝
StopIteration raised
상태: GEN_CLOSED


In [15]:
def gen_nums():
    for i in range(int(1e6)):
        yield i
        
nums = gen_nums()

# 제너레이터 표현식
nums = (i for i in range(int(1e6)))

In [19]:
# 제너레이터 주의사항
def generator_func():
    print('1번 항목 처리')
    yield 1  # 1 반환후 대기
    print('2번 항목 처리')
    yield 2  # 2 반환후 대기

gen = generator_func()
print(next(gen)) # 1번 항목 처리 1
print(next(gen)) # 2번 항목 처리 2
print(next(gen)) # StopIteration

1번 항목 처리
1
2번 항목 처리
2


StopIteration: 

In [20]:
# 재사용 불가능한 제너레이터
for i in gen:
    print(i) # 아무것도 출력되지 않음

# 재할당하여 다시 제너레이터를 사용
gen = generator_func()
for i in gen:
    print(i)

1번 항목 처리
1
2번 항목 처리
2
